# 🚀 End-to-End Extraction, Segmentation, and AI Validation Pipeline
This notebook executes the entire zero-shot pipeline (Grounding DINO + SAM 2) and introduces a novel **VLM-as-a-Judge (Qwen3-VL)** step. 

Instead of manual QA, we pass the resulting segmentation masks back into the Vision Language Model and ask it to critique the segmentation quality, identifying background leakage, missing feathers, or clipped boundaries.

### Step 1: Environment Setup and Initialization
We import the necessary dependencies for computer vision (`cv2`, `PIL`), tensor computation (`torch`, `numpy`), and AI model execution (`transformers`, `ultralytics`, `mlx_vlm`). We also configure PyTorch to use Apple's Metal Performance Shaders (MPS) for hardware acceleration.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM
from mlx_vlm import load, generate

# Configure hardware accelerator to use Apple Silicon (MPS)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Hardware Accelerator: {device}")

# Define paths for reading raw data and saving processed outputs
DATA_DIR = "../data/raw"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Select the test image
image_path = os.path.join(DATA_DIR, 'A1383 2000-im1316.jpg')

### Step 2: Zero-Shot Object Detection and Segmentation
This pipeline operates in two sequential stages without requiring any prior model training:
1. **Grounding DINO:** Uses a text prompt (`"bird feather."`) to identify regions of interest and output bounding box coordinates.
2. **SAM 2 (Segment Anything Model):** Uses those bounding box coordinates as spatial prompts to mathematically isolate the pixels and generate a segmentation mask for each object.

In [ ]:
def run_pipeline(img_path):
    print("1. Running Grounding DINO for bounding box detection...")
    dino_id = 'IDEA-Research/grounding-dino-base'
    dino_processor = AutoProcessor.from_pretrained(dino_id)
    dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)
    
    img = Image.open(img_path).convert('RGB')
    img_area = img.size[0] * img.size[1]
    
    inputs = dino_processor(images=img, text='bird feather.', return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = dino_model(**inputs)
    
    results = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, target_sizes=[img.size[::-1]]
    )[0]
    
    boxes = []
    for score, box in zip(results['scores'], results['boxes']):
        # Lower threshold to 0.25 to catch low-contrast feathers (like the central grey one)
        if score > 0.25:
            b = box.tolist()
            w = b[2] - b[0]
            h = b[3] - b[1]
            
            # 1. Reject massive overlapping boxes (> 20% of image area)
            if (w * h) > (img_area * 0.20):
                continue
                
            # 2. Shrink boxes by 5% to force SAM to focus on foreground and stop background bleed
            shrink = 0.05
            b[0] += w * shrink
            b[1] += h * shrink
            b[2] -= w * shrink
            b[3] -= h * shrink
            
            boxes.append(b)
            
    del dino_model
    torch.mps.empty_cache()
    
    print(f"   Filtered down to {len(boxes)} high-quality bounding boxes. Passing to SAM 2...")
    
    sam_model = SAM('sam2.1_b.pt')
    sam_results = sam_model(img_path, bboxes=boxes, device='mps', verbose=False, retina_masks=True)[0]
    
    return boxes, sam_results

boxes, sam_results = run_pipeline(image_path)

### Step 3: Visualization and Composite Generation
To evaluate the accuracy of the segmentation, we use OpenCV to overlay the AI-generated artifacts onto the original image.
*   **Bright Green Rectangles:** Bounding boxes from Grounding DINO.
*   **Red Overlays:** Pixel masks generated by SAM 2.

This composite image is then saved to disk for the VLM to analyze.

In [ ]:
def create_validation_composite(img_path, boxes, sam_res):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img_rgb.copy()
    
    # Apply the segmentation masks as a red overlay
    if sam_res.masks is not None:
        for m in sam_res.masks.data.cpu().numpy():
            # Resize the mask array to match the original image dimensions
            m = cv2.resize(m.astype(np.float32), (img.shape[1], img.shape[0]))
            overlay[m > 0.5] = [255, 0, 0] # Highlight mask area in pure red
            
    # Draw the bounding boxes in bright green
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 6)
        
    # Blend the overlay with the original image using a weighted sum
    blended = cv2.addWeighted(img_rgb, 0.4, overlay, 0.6, 0)
    
    comp_path = os.path.join(OUTPUT_DIR, "validation_composite.jpg")
    cv2.imwrite(comp_path, cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))
    
    # Display the result using matplotlib
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(blended)
    ax.set_title("AI Quality Assurance Composite (Masks & Bounds)")
    ax.axis('off')
    plt.show()
    
    return comp_path

composite_path = create_validation_composite(image_path, boxes, sam_results)

### Step 4: VLM-as-a-Judge for Quality Assurance
Instead of manual review, we use Qwen3-VL to evaluate the composite image. We prompt the model to look for specific failure modes common in zero-shot segmentation pipelines (e.g., background leakage, missing objects, grouped bounding boxes). The model enforces structured output formatting by returning its critique as a JSON object.

In [ ]:
def vlm_as_a_judge(comp_img_path):
    print("Loading Qwen3-VL to judge segmentation quality...")
    vlm_path = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
    model, processor = load(vlm_path)
    
    # Define the QA criteria and enforce JSON output format
    raw_prompt = '''
    You are an expert computer vision Quality Assurance bot. 
    Analyze the provided image. The bright green boxes represent bounding box detections. The red shaded areas represent pixel-perfect segmentation masks generated by SAM 2.
    
    Critique the quality of the segmentation. 
    1. Are the red masks covering the bird feathers entirely? Look closely, did it miss the middle feather?
    2. Is there background leakage (red mask bleeding onto the white paper, ruler, or tape)?
    3. Are the green boxes wrapping multiple feathers instead of individual ones?
    4. Provide an overall confidence score out of 10.
    
    Return EXACTLY in this JSON format:
    {
      "all_feathers_covered": true/false,
      "background_leakage_detected": true/false,
      "green_boxes_grouped_feathers": true/false,
      "quality_score_1_to_10": 9,
      "critique": "Your brief explanation here."
    }
    '''
    
    print("Asking VLM to critique the composite...")
    try:
        # Apply ChatML template formatting required by Qwen3-VL Instruct
        chatml_prompt = f"<|im_start|>system\nYou are a precise computer vision analyst.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>\n{raw_prompt}<|im_end|>\n<|im_start|>assistant\n"
        
        output = generate(model, processor, prompt=chatml_prompt, image=[comp_img_path], max_tokens=256)
        output_text = getattr(output, 'text', str(output))
        
        # Extract the JSON block using regular expressions
        import re
        match = re.search(r'\{.*?\}', output_text, re.DOTALL)
        
        print("\n=== VLM JUDGE REPORT ===")
        if match:
            print(match.group(0))
        else:
            print(output_text)
    except Exception as e:
        print(f"VLM evaluation failed: {e}")

vlm_as_a_judge(composite_path)